Нобелевские лауреаты 1901-2022: доля США до и после 1945 года

Данные

In [ ]:
import pandas as pd
import numpy as np
import glob

files = sorted(glob.glob("Nobel*.csv"))
cats = ["Chemistry", "Economics", "Literature", "Medicine", "Physics", "Peace"]

frames = []
for f in files:
    t = pd.read_csv(f)
    t["category"] = next(c for c in cats if c in f)
    frames.append(t[["Year", "Laureate", "Country", "category"]])

df = pd.concat(frames, ignore_index=True)
print(df.shape)
print(df["category"].value_counts())
print(df.isna().sum())

(988, 4)
category
Medicine      225
Physics       222
Chemistry     191
Peace         139
Literature    119
Economics      92
Name: count, dtype: int64
Year        0
Laureate    0
Country     0
category    0
dtype: int64


In [ ]:
df["Country"] = (df["Country"]
                 .str.replace(r"\[.*?\]", "", regex=True)
                 .str.replace(r"\(.*?\)", "", regex=True)
                 .str.replace("\xa0", " ", regex=False)
                 .str.strip())

print(df["Country"].value_counts().head(8))

Country
United States     309
United Kingdom     99
France             58
Germany            47
Sweden             29
Switzerland        27
Japan              25
West Germany       24
Name: count, dtype: int64


Гипотеза

H0: доля лауреатов из США до и после 1945 года одинаковая. H1: доли разные. Уровень значимости 0.05.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

df["usa"] = df["Country"].str.contains("United States", na=False).astype(int)
df["period"] = np.where(df["Year"] < 1945, "до 1945", "с 1945")

g = df.groupby("period")["usa"].agg(["sum", "count"])
g["share"] = g["sum"] / g["count"]
print(g)

z_stat, p_val = proportions_ztest(g["sum"], g["count"])
print("z =", round(z_stat, 2), " p-value =", p_val)

for period, row in g.iterrows():
    low, high = proportion_confint(row["sum"], row["count"])
    print(period, "доля:", round(row["share"], 3), " 95% ДИ:", round(low, 3), round(high, 3))

         sum  count     share
period                       
до 1945   28    217  0.129032
с 1945   364    771  0.472114
z = -9.13  p-value = 7.126090022989213e-20
до 1945 доля: 0.129  95% ДИ: 0.084 0.174
с 1945 доля: 0.472  95% ДИ: 0.437 0.507


до 1945: 28 из 217 (12.9%), ДИ 8.4% - 17.4%

с 1945: 364 из 771 (47.2%), ДИ 43.7% - 50.7%

z = -9.13, p-value = 7e-20, H0 отвергается

SQL

In [ ]:
import sqlite3

con = sqlite3.connect("nobel.db")
df.to_sql("laureates", con, index=False, if_exists="replace")

988

Лауреаты по десятилетиям

In [ ]:
q1 = '''
SELECT (Year / 10) * 10 AS decade, COUNT(*) AS n
FROM laureates
GROUP BY decade
ORDER BY decade
'''
pd.read_sql(q1, con)

,decade,n
0,1900,57
1,1910,40
2,1920,54
3,1930,56
4,1940,42
5,1950,72
6,1960,79
7,1970,104
8,1980,97
9,1990,104


Топ-5 стран

In [ ]:
q2 = '''
SELECT Country, COUNT(*) AS n
FROM laureates
GROUP BY Country
ORDER BY n DESC
LIMIT 5
'''
pd.read_sql(q2, con)

,Country,n
0,United States,309
1,United Kingdom,99
2,France,58
3,Germany,47
4,Sweden,29


Категории, где с 1945 года больше 100 лауреатов

In [ ]:
q3 = '''
SELECT category, COUNT(*) AS total
FROM laureates
WHERE Year >= 1945
GROUP BY category
HAVING COUNT(*) > 100
ORDER BY total DESC
'''
pd.read_sql(q3, con)

,category,total
0,Medicine,179
1,Physics,174
2,Chemistry,149


Доля США по категориям с 1945 года

In [ ]:
q4 = '''
SELECT category,
       SUM(usa) AS usa_laureates,
       COUNT(*) AS total,
       ROUND(SUM(usa) * 100.0 / COUNT(*), 1) AS usa_share
FROM laureates
WHERE Year >= 1945
GROUP BY category
ORDER BY usa_share DESC
'''
pd.read_sql(q4, con)

,category,usa_laureates,total,usa_share
0,Economics,68,92,73.9
1,Medicine,102,179,57.0
2,Physics,91,174,52.3
3,Chemistry,78,149,52.3
4,Peace,16,97,16.5
5,Literature,9,80,11.3


Выводы

После 1945 года доля лауреатов из США выросла с 12.9% до 47.2% (p-value 7e-20), доверительные интервалы не пересекаются. Больше всего американцев в экономике (73.9%), меньше всего в литературе (11.3%).

Ограничения:

несколько стран в одной строке Country считаются как США

в категориях мало наблюдений

причины роста не изучались